# Github

In [3]:
import os
from pathlib import Path

repo_root = Path.cwd()
print(f"Working locally in: {repo_root}")


Working locally in: /Users/shainakumar/cs6501workshop-1/Topic7MCP


# Set Up

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

if not os.getenv("ASTA_API_KEY"):
    os.environ["ASTA_API_KEY"] = getpass("ASTA_API_KEY: ")

print("API keys loaded for this notebook session")


API keys loaded for this notebook session


In [9]:
%pip install openai requests


  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 2.7 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openai]2m7/8 [openai]c]
Note: you may need to restart the kernel to use updated packages.


## Exercise A

In [ ]:
import requests
import os
import json

URL = "https://asta-tools.allen.ai/mcp/v1"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "x-api-key": os.environ["ASTA_API_KEY"]
}

payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

resp = requests.post(URL, headers=headers, json=payload)
text = resp.text

# extract the JSON after "data:"
for line in text.splitlines():
    if line.startswith("data:"):
        data = line.replace("data:", "").strip()
        resp_json = json.loads(data)
        break

tools = resp_json["result"]["tools"]

for tool in tools:
    print(f"\nTool: {tool['name']}")
    print(f"  Description: {tool['description']}")

    schema = tool["inputSchema"]
    props = schema.get("properties", {})
    required = schema.get("required", [])

    for name, meta in props.items():
        t = meta.get("type", "unknown")
        if name in required:
            print(f"  Required: {name} ({t})")
        else:
            print(f"  Optional: {name} ({t})")


Tool: get_paper
  Description: 
Get details about a paper by its id.

Args:
    paper_id: The id of the paper to get. The following types of IDs are supported:
        <sha> - a Semantic Scholar ID, e.g. 649def34f8be52c8b66281af98ae884c09aef38b
        CorpusId:<id> - a Semantic Scholar numerical ID, e.g. CorpusId:215416146
        DOI:<doi> - a Digital Object Identifier, e.g. DOI:10.18653/v1/N18-3011
        ARXIV:<id> - arXiv.rg, e.g. ARXIV:2106.15928
        MAG:<id> - Microsoft Academic Graph, e.g. MAG:112218234
        ACL:<id> - Association for Computational Linguistics, e.g. ACL:W12-3903
        PMID:<id> - PubMed/Medline, e.g. PMID:19872477
        PMCID:<id> - PubMed Central, e.g. PMCID:2323736
        URL:<url> - URL from one of the sites listed below, e.g. URL:https://arxiv.org/abs/2106.15928v1

    fields: String of comma-separated fields to include in the response. E.g "url,year,authors".
    Default is "title". Available fields are: abstract, authors, citations, fieldsOf

## Exercise B

In [ ]:
import requests
import os
import json

URL = "https://asta-tools.allen.ai/mcp/v1"

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "x-api-key": os.environ["ASTA_API_KEY"]
}


def parse_sse_json(resp):
    """
    Extract JSON from MCP text/event-stream response
    """
    for line in resp.text.splitlines():
        if line.startswith("data:"):
            return json.loads(line.replace("data:", "").strip())

    raise ValueError("No JSON found in MCP response")


def call_tool(name, arguments, req_id=1):

    payload = {
        "jsonrpc": "2.0",
        "id": req_id,
        "method": "tools/call",
        "params": {
            "name": name,
            "arguments": arguments
        }
    }

    resp = requests.post(URL, headers=headers, json=payload)
    data = parse_sse_json(resp)

    # Asta returns JSON embedded as text
    content = data["result"]["content"]

    return content

def drill1():

    papers = call_tool(
        "search_papers_by_relevance",
        {
            "keyword": "large language model agents",
            "fields": "title,abstract,year,authors",
            "limit": 5
        },
        2
    )

    print("\nTop LLM Agent Papers:")

    for i, _ in enumerate(papers):
        p = json.loads(papers[i]["text"])
        print(f"{i+1}. {p["title"]} ({p["year"]})")


def drill2():

    citations = call_tool(
        "get_citations",
        {
            "paper_id": "ARXIV:1810.04805",
            "fields": "title,year",
            "publication_date_range": "2023-01-01:",
            "limit": 10
        },
        3
    )

    print("\nNumber of citations since 2023:", len(citations))

    for i, _ in enumerate(citations[:5]):
        p = json.loads(citations[i]["text"])
        print(f"{i+1}. {p["citingPaper"]["title"]}")


def drill3():

    refs = call_tool(
        "get_paper",
        {
            "paper_id": "ARXIV:2210.03629",
            "fields": "references,references.title,references.year"
        },
        4
    )

    papers = json.loads(refs[0]["text"])
    references = sorted([p for p in papers["references"] if p["year"] is not None], key=lambda x: x["year"])
    print("\nReferences for ReAct:")

    for p in references:
        print(p["year"], "-", p["title"])


if __name__ == "__main__":
    drill1()
    drill2()
    drill3()


Top LLM Agent Papers:
1. InjecAgent: Benchmarking Indirect Prompt Injections in Tool-Integrated Large Language Model Agents (2024)
2. Memory-R1: Enhancing Large Language Model Agents to Manage and Utilize Memories via Reinforcement Learning (2025)
3. A Survey of Large Language Model Agents for Question Answering (2025)
4. Emergence of human-like polarization among large language model agents (2025)
5. PersonaAgent: When Large Language Model Agents Meet Personalization at Test Time (2025)

Number of citations since 2023: 10
1. Enhancing large language models for knowledge graph question answering via multi-granularity knowledge injection and structured reasoning path-augmented prompting
2. Multi-view dynamic perception framework for Chinese harmful meme detection
3. SEGA: Selective cross-lingual representation via sparse guided attention for low-resource multilingual named entity recognition
4. Validating generative agent-Based modeling in social media simulations through the lens of t

## Exercise C

In [ ]:
import requests
import os
import json
from openai import OpenAI

URL = "https://asta-tools.allen.ai/mcp/v1"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "x-api-key": os.environ["ASTA_API_KEY"]
}


def parse_sse_json(resp):
    """
    Extract JSON from MCP text/event-stream response
    """
    for line in resp.text.splitlines():
        if line.startswith("data:"):
            return json.loads(line.replace("data:", "").strip())

    raise ValueError("No JSON found in MCP response")


def get_asta_tools():

    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/list",
        "params": {}
    }

    resp = requests.post(URL, headers=headers, json=payload)
    data = parse_sse_json(resp)

    tools = data["result"]["tools"]

    return [
        {
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t["description"],
                "parameters": t["inputSchema"]
            }
        }
        for t in tools
    ]


def call_asta_tool(name, arguments):
    payload = {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": name,
            "arguments": arguments
        }
    }

    resp = requests.post(URL, headers=headers, json=payload)
    data = parse_sse_json(resp)

    # Asta returns JSON embedded as text
    content = data["result"]["content"]
    parts = [c["text"] for c in content]
    return "\n---\n".join(parts)


def chat(user_message, messages, tools):

    while True:
        if not user_message:
            user_message = input("\nUser: ")

        messages.append({"role": "user", "content": user_message})

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return msg.content

        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"Calling: {name}")
            print(f"Args: {args}")
            result = call_asta_tool(name, args)

            if len(result) > 8000:
                result = result[:8000] + "\n... [truncated]"

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })


def main():
    print("=" * 60)
    print("Asta Research Chatbot")
    print("=" * 60)

    tools = get_asta_tools()
    print(f"Loaded {len(tools)} tools")

    messages = [
      {
        "role": "system",
        "content": """
        You are a research assistant with access to Semantic Scholar via Asta tools.

        When calling tools:
        - Use ONLY parameters defined in the schema.
        - Never invent parameter names.
        - If unsure, omit optional fields.
        """
      }
    ]

    test_queries = [
        "Find recent papers about large language model agents",
        "Who wrote Attention is All You Need and what else have they published?",
        "What papers cite the original BERT paper (ARXIV:1810.04805)?",
        "Summarize the references used in the ReAct paper (ARXIV:2210.03629)",
    ]

    if len(test_queries) > 0:
        for query in test_queries:
            print(f"\n{'─' * 60}")
            print(f"Query: {query}")
            print(f"{'─' * 60}\n")

            answer = chat(query, messages, tools)
            print(f"\nAnswer:\n")
            for line in answer.split("\n"):
                print(f"  {line}")
            print()
    else:
        while True:
            print(f"\n{'─' * 60}")
            query = input("Query: ")
            print(f"{'─' * 60}\n")

            answer = chat(query, messages, tools)
            print(f"\nAnswer:\n")
            for line in answer.split("\n"):
                print(f"  {line}")
            print()

if __name__ == "__main__":
    main()

Asta Research Chatbot
Loaded 8 tools

────────────────────────────────────────────────────────────
Query: Find recent papers about large language model agents
────────────────────────────────────────────────────────────

Calling: search_papers_by_relevance
Args: {'keyword': 'large language model agents', 'limit': 5, 'fields': 'title,authors,journal,publicationDate,url'}

Answer:

  Here are some recent papers related to large language model agents:
  
  1. **[InjecAgent: Benchmarking Indirect Prompt Injections in Tool-Integrated Large Language Model Agents](https://www.semanticscholar.org/paper/c8eee9766f0968e8f1b1be0731bc70b85be0ac97)**  
     - **Authors:** Qiusi Zhan, Zhixiang Liang, Zifan Ying, Daniel Kang  
     - **Publication Date:** March 5, 2024  
     - **Journal:** ArXiv
    
  2. **[Memory-R1: Enhancing Large Language Model Agents to Manage and Utilize Memories via Reinforcement Learning](https://www.semanticscholar.org/paper/9df0947ec8a14c43e69f3b72ece5e9af2b8e8215)**  
  

## Exercise D

In [ ]:
import requests
import os
import json
import sys
from openai import OpenAI

URL = "https://asta-tools.allen.ai/mcp/v1"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "x-api-key": os.environ["ASTA_API_KEY"]
}


def parse_sse_json(resp):
    """Extract JSON from MCP text/event-stream response."""
    for line in resp.text.splitlines():
        if line.startswith("data:"):
            return json.loads(line.replace("data:", "").strip())
    raise ValueError("No JSON found in MCP response")


def call_tool(name, arguments, req_id=1):
    payload = {
        "jsonrpc": "2.0",
        "id": req_id,
        "method": "tools/call",
        "params": {"name": name, "arguments": arguments}
    }
    resp = requests.post(URL, headers=headers, json=payload)
    resp.raise_for_status()
    data = parse_sse_json(resp)
    if "error" in data:
        raise RuntimeError(f"MCP error from {name}: {data['error']}")
    return data["result"]


def content_as_json_items(result):
    items = []
    for item in result.get("content", []):
        text = item.get("text", "").strip()
        if not text:
            continue
        if text.startswith("Error executing tool"):
            raise RuntimeError(text)
        items.append(json.loads(text))
    return items


def result_payload(result):
    structured = result.get("structuredContent", {}).get("result")
    if structured is not None:
        return structured
    items = content_as_json_items(result)
    if not items:
        raise ValueError(f"No parseable tool payload found in result: {result}")
    return items[0] if len(items) == 1 else items


def get_paper_payload(paper_id, fields):
    return result_payload(call_tool("get_paper", {"paper_id": paper_id, "fields": fields}))


def search_react_payload(fields):
    result = call_tool(
        "search_paper_by_title",
        {
            "title": "ReAct: Synergizing Reasoning and Acting in Language Models",
            "fields": fields,
        },
    )
    payload = result_payload(result)
    return payload[0] if isinstance(payload, list) else payload


def get_seed_payload(seed_id, fields):
    candidates = [seed_id]
    if seed_id == "ARXIV:2210.03629":
        candidates.append("URL:https://arxiv.org/abs/2210.03629")

    last_error = None
    for candidate in candidates:
        try:
            return get_paper_payload(candidate, fields)
        except RuntimeError as exc:
            last_error = exc

    if seed_id == "ARXIV:2210.03629":
        print(f"Paper ID lookup failed: {last_error}")
        print("Falling back to title search.")
        return search_react_payload(fields)

    raise last_error


def generate_report(seed, references, citations, author_profiles):
    context = json.dumps({
        "seed_paper": {
            "title": seed["title"],
            "abstract": seed.get("abstract"),
            "year": seed.get("year"),
            "authors": [a["name"] for a in seed.get("authors", [])],
            "fields": seed.get("fieldsOfStudy"),
            "citations": seed.get("citationCount"),
        },
        "key_references": [
            {"title": r["title"], "year": r.get("year"),
             "abstract": (r.get("abstract") or "")[:200],
             "citations": r.get("citationCount")}
            for r in references
        ],
        "recent_citations": [
            {"title": c["title"], "year": c.get("year"),
             "abstract": (c.get("abstract") or "")[:200],
             "citations": c.get("citationCount")}
            for c in citations
        ],
        "author_profiles": [
            {"name": a["name"],
             "notable_work": a["top_paper"]["title"] if a["top_paper"] else None,
             "notable_citations": a["top_paper"].get("citationCount") if a["top_paper"] else None}
            for a in author_profiles
        ],
    }, indent=2)

    prompt = f"""Based on the given research data, generate a structured markdown report that contains the following:

1. **Summary** — A one-paragraph summary of the seed paper
2. **Foundational Works** — The top 5 references with title, year, and significance
3. **Recent Developments** — The top 5 most recent citing papers with title, year, and significance
4. **Author Profiles** — Each author's name and their most notable other work

Research data:
{context}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an academic research analyst. Write clear, precise markdown reports about scientific papers."},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content


def run(seed_id):
    seed_fields = "title,abstract,year,authors,fieldsOfStudy,citationCount"
    s = get_seed_payload(seed_id, seed_fields)

    ref_fields = "references,references.title,references.year,references.abstract,references.citationCount"
    paper = get_seed_payload(seed_id, ref_fields)
    references = sorted(
        [p for p in paper.get("references", []) if p.get("citationCount") is not None],
        key=lambda x: x["citationCount"],
        reverse=True,
    )[:5]

    cites = call_tool(
        "get_citations",
        {
            "paper_id": seed_id,
            "fields": "title,year,abstract,citationCount",
            "publication_date_range": "2023-01-01:",
            "limit": 5,
        }
    )
    citation_results = result_payload(cites)
    c = [item["citingPaper"] for item in citation_results[:5]]

    a = []
    for author in s.get("authors", []):
        papers = call_tool(
            "get_author_papers",
            {"author_id": author["authorId"], "paper_fields": "title,year,citationCount"}
        )
        works = sorted(
            result_payload(papers),
            key=lambda x: x.get("citationCount") or 0,
            reverse=True,
        )
        if works and works[0]["title"] == s["title"] and len(works) > 1:
            top_paper = works[1]
        else:
            top_paper = works[0] if works else None
        a.append({"name": author["name"], "top_paper": top_paper})

    return generate_report(s, references, c, a)


if __name__ == "__main__":
    seed_id = "ARXIV:2210.03629"
    if len(sys.argv) > 1 and sys.argv[1].startswith(("ARXIV:", "DOI:", "CorpusId:", "PMID:", "PMCID:", "URL:")):
        seed_id = sys.argv[1]

    print(f"Seed paper: {seed_id}")
    print("\n" + "=" * 60)
    print("AGENT REPORT")
    print("=" * 60 + "\n")
    print(run(seed_id))

